<a href="https://colab.research.google.com/github/MMujtabaX/Boosting/blob/main/Boosting_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Boosting: AdaBoost · Gradient Boosting · XGBoost
### AtomCamp MLOps Series

> **Sequential learners that fix each other's mistakes.**  
> Each new model is trained specifically to correct the errors of the one before it.

---
**What you'll learn in this notebook:**
1. Boosting vs Bagging — the core difference
2. AdaBoost — algorithm, math, and visualization
3. Gradient Boosting — residuals step by step
4. XGBoost (via sklearn GBM) — hyperparameter tuning
5. Comparing all three on the same dataset


## 0. Imports & Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import make_classification, make_moons, load_breast_cancer
from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier,
    RandomForestClassifier, BaggingClassifier
)
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

np.random.seed(42)
print('All libraries loaded successfully!')


---
## 1. Weak Learners — The Building Block

A **weak learner** is a model that does just slightly better than random guessing.  
In boosting, the weak learner is almost always a **Decision Stump** — a tree with depth=1.

Let's see how bad a single stump is, and how combining them becomes powerful.


In [ ]:
# Create a classification dataset
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Single stump
stump = DecisionTreeClassifier(max_depth=1)
stump.fit(X_train, y_train)
stump_acc = accuracy_score(y_test, stump.predict(X_test))

# Print the stump rules
print('Decision Stump Rules:')
print(export_text(stump, feature_names=['X1', 'X2']))
print(f'Stump Accuracy: {stump_acc:.3f}  (barely above 50% random guessing)')


In [ ]:
# Visualize: single stump vs boosted ensemble
def plot_decision_boundary(model, X, y, title, ax):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='k', s=40)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

adaboost_100 = AdaBoostClassifier(n_estimators=100, random_state=42)
adaboost_100.fit(X_train, y_train)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_decision_boundary(stump, X_train, y_train,
    f'Single Stump\nAcc={stump_acc:.2f}', axes[0])

for n_est, ax in zip([10, 100], axes[1:]):
    m = AdaBoostClassifier(n_estimators=n_est, random_state=42).fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    plot_decision_boundary(m, X_train, y_train,
        f'AdaBoost ({n_est} stumps)\nAcc={acc:.2f}', ax)

plt.suptitle('From Weak to Strong: Adding More Stumps', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('Notice how the boundary gets more complex and accurate as we add more stumps!')


---
## 2. AdaBoost — Step by Step

**The two core ideas in AdaBoost:**
1. **Sample weights** — misclassified points get higher weight → next learner focuses on them
2. **Learner weights (α)** — accurate learners get more say in the final vote

### Key Formulas

| Step | Formula | Meaning |
|------|---------|--------|
| Initialize | $w_i = 1/N$ | All samples start equal |
| Error | $\varepsilon = \sum w_i \cdot \mathbb{1}[\text{wrong}]$ | Weighted error rate |
| Learner weight | $\alpha = 0.5 \times \ln\left(\frac{1-\varepsilon}{\varepsilon}\right)$ | How much to trust this learner |
| Update weights | $w_i \leftarrow w_i \times e^{-\alpha y_i h_t(x_i)}$ | Upweight wrong, downweight right |
| Final prediction | $\hat{y} = \text{sign}\left(\sum_t \alpha_t h_t(x)\right)$ | Weighted vote |


In [ ]:
# Visualize the alpha (learner weight) formula
epsilon = np.linspace(0.01, 0.99, 200)
alpha = 0.5 * np.log((1 - epsilon) / epsilon)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Alpha curve
axes[0].plot(epsilon, alpha, color='#0D8E8E', linewidth=2.5)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Random guess (ε=0.5)')
axes[0].fill_between(epsilon, alpha, 0, where=(epsilon < 0.5), alpha=0.15, color='green', label='Better than random → α > 0')
axes[0].fill_between(epsilon, alpha, 0, where=(epsilon > 0.5), alpha=0.15, color='red', label='Worse than random → α < 0')
axes[0].set_xlabel('Error rate ε', fontsize=12)
axes[0].set_ylabel('Learner weight α', fontsize=12)
axes[0].set_title('α = 0.5 × ln((1−ε)/ε)', fontsize=13, fontweight='bold')
axes[0].legend()

# Specific alpha values table
eps_vals = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
alpha_vals = [0.5 * np.log((1-e)/e) for e in eps_vals]
colors = ['#2ECC71' if a > 0 else '#E74C3C' for a in alpha_vals]

axes[1].barh(range(len(eps_vals)), alpha_vals, color=colors, edgecolor='k', alpha=0.8)
axes[1].set_yticks(range(len(eps_vals)))
axes[1].set_yticklabels([f'ε={e}' for e in eps_vals])
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('α value', fontsize=12)
axes[1].set_title('Learner Weight by Error Rate', fontsize=13, fontweight='bold')
for i, (a, c) in enumerate(zip(alpha_vals, colors)):
    axes[1].text(a + 0.02 if a >= 0 else a - 0.02, i, f'{a:.2f}',
                 va='center', ha='left' if a >= 0 else 'right', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Visualize sample weight evolution across rounds
from sklearn.ensemble import AdaBoostClassifier

# Simple 2D dataset
np.random.seed(42)
X_simple = np.array([[1,2],[2,3],[3,1],[4,4],[5,2],[6,3],[7,1],[8,3]])
y_simple  = np.array([0,0,0,0,1,1,1,1])

# Manually simulate one round of AdaBoost
N = len(y_simple)
weights = np.ones(N) / N

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for round_num, ax in enumerate(axes):
    stump_r = DecisionTreeClassifier(max_depth=1)
    stump_r.fit(X_simple, y_simple, sample_weight=weights)
    preds = stump_r.predict(X_simple)
    wrong = (preds != y_simple).astype(int)
    eps = np.sum(weights * wrong)
    alpha_r = 0.5 * np.log((1 - eps + 1e-10) / (eps + 1e-10))

    # Plot
    bar_colors = ['#E05C5C' if w else '#0D8E8E' for w in wrong]
    bars = ax.bar(range(N), weights * 100, color=bar_colors, edgecolor='k', alpha=0.85)
    ax.set_xticks(range(N))
    ax.set_xticklabels([f'S{i+1}' for i in range(N)])
    ax.set_ylabel('Weight (%)')
    ax.set_title(f'Round {round_num+1}\nε={eps:.2f}, α={alpha_r:.2f}', fontweight='bold')

    correct_patch = mpatches.Patch(color='#0D8E8E', label='Correct')
    wrong_patch   = mpatches.Patch(color='#E05C5C', label='Wrong (upweighted next round)')
    ax.legend(handles=[correct_patch, wrong_patch], fontsize=8)

    # Update weights for next round
    weights *= np.exp(-alpha_r * y_simple * (2 * preds - 1))
    weights /= weights.sum()

plt.suptitle('Sample Weight Evolution Across AdaBoost Rounds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# AdaBoost on Breast Cancer dataset — real use case
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Train with different number of estimators
n_estimators_range = [1, 5, 10, 20, 50, 100, 200]
train_accs, test_accs = [], []

for n in n_estimators_range:
    model = AdaBoostClassifier(n_estimators=n, random_state=42)
    model.fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, model.predict(X_tr)))
    test_accs.append(accuracy_score(y_te, model.predict(X_te)))

plt.figure(figsize=(10, 5))
plt.plot(n_estimators_range, train_accs, 'o-', label='Train Accuracy', color='#0D8E8E', linewidth=2)
plt.plot(n_estimators_range, test_accs,  's-', label='Test Accuracy',  color='#E05C5C', linewidth=2)
plt.xlabel('Number of Estimators (stumps)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('AdaBoost: Accuracy vs Number of Stumps (Breast Cancer)', fontsize=13, fontweight='bold')
plt.legend()
plt.ylim(0.85, 1.01)
plt.grid(alpha=0.3)
plt.show()

best_n = n_estimators_range[np.argmax(test_accs)]
print(f'Best number of estimators: {best_n}')
print(f'Best test accuracy: {max(test_accs):.4f}')


---
## 3. Gradient Boosting — Fitting the Residuals

Instead of reweighting samples, Gradient Boosting trains each new tree directly on the **residual errors** (how wrong we were).

### Update Rule:
$$F_0(x) = \bar{y} \quad \text{(initial prediction = mean)}$$
$$r_{ti} = -\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \quad \text{(pseudo-residuals)}$$
$$F_t(x) = F_{t-1}(x) + \eta \cdot h_t(x) \quad \text{(additive update)}$$

For **squared error loss**: pseudo-residual = actual − predicted (simple residual).

For **log-loss (classification)**: pseudo-residual = actual − predicted probability.


In [ ]:
# MANUALLY simulate Gradient Boosting on a toy regression problem
# So you can see exactly what's happening at each step

from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
X_reg = np.sort(np.random.uniform(0, 10, 50)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + np.random.normal(0, 0.3, 50)

# Step 0: Initial prediction = mean
F = np.full(len(y_reg), y_reg.mean())
lr = 0.5
n_rounds = 5

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

# Round 0: just the mean
axes[0].scatter(X_reg, y_reg, color='steelblue', s=30, label='True values')
axes[0].hlines(y_reg.mean(), 0, 10, colors='red', linewidth=2, label=f'F0 = mean = {y_reg.mean():.2f}')
residuals_0 = y_reg - F
axes[0].set_title(f'Round 0: Initial Prediction (mean)\nRSS={np.sum(residuals_0**2):.2f}', fontweight='bold')
axes[0].legend(fontsize=8)

for t in range(1, n_rounds + 1):
    residuals = y_reg - F               # compute residuals
    tree = DecisionTreeRegressor(max_depth=2)
    tree.fit(X_reg, residuals)           # fit tree TO RESIDUALS
    update = tree.predict(X_reg)
    F = F + lr * update                  # update prediction

    ax = axes[t]
    ax.scatter(X_reg, y_reg, color='steelblue', s=30, alpha=0.6, label='True')
    ax.plot(X_reg, F, color='red', linewidth=2, label=f'F{t}')

    # Show residuals as vertical lines
    for xi, yi, fi in zip(X_reg.ravel(), y_reg, F):
        ax.plot([xi, xi], [fi, yi], color='orange', alpha=0.3, linewidth=1)

    new_residuals = y_reg - F
    ax.set_title(f'Round {t}: After adding Tree {t}\nRSS={np.sum(new_residuals**2):.2f}', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Gradient Boosting: Residuals Shrink Each Round (orange lines = current errors)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print('Notice how RSS (sum of squared errors) drops with each round!')


In [ ]:
# Effect of learning rate on Gradient Boosting
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error

X_r, y_r = make_regression(n_samples=500, n_features=10, noise=20, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

learning_rates = [0.01, 0.05, 0.1, 0.3, 1.0]
n_est = 200

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#0D8E8E', '#6D3FA0', '#F0A500', '#E05C5C', '#16A34A']

for lr_val, color in zip(learning_rates, colors):
    train_errors, test_errors = [], []
    for n in range(1, n_est+1, 5):
        gbm = GradientBoostingClassifier if False else GradientBoostingClassifier
        model = GradientBoostingClassifier(n_estimators=n, learning_rate=lr_val,
                                            max_depth=3, random_state=42)
        # Use regression model instead
        from sklearn.ensemble import GradientBoostingRegressor
        m = GradientBoostingRegressor(n_estimators=n, learning_rate=lr_val,
                                      max_depth=3, random_state=42)
        m.fit(Xr_tr, yr_tr)
        test_errors.append(np.sqrt(mean_squared_error(yr_te, m.predict(Xr_te))))
    ax.plot(range(1, n_est+1, 5), test_errors, label=f'lr={lr_val}', color=color, linewidth=2)

ax.set_xlabel('Number of Trees', fontsize=12)
ax.set_ylabel('Test RMSE', fontsize=12)
ax.set_title('Learning Rate Trade-off:\nLow lr = slower convergence but better final performance', fontsize=13, fontweight='bold')
ax.legend(title='Learning Rate')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Key insight: Low lr + many trees almost always beats high lr + few trees')


---
## 4. XGBoost-style Tuning with sklearn GradientBoosting

> **Note:** XGBoost is not installed in this environment, so we use sklearn's `GradientBoostingClassifier`  
> which implements the same algorithm. In practice, swap it with `xgboost.XGBClassifier` — same parameters.

### XGBoost's extra objective function:
$$\text{Obj} = \sum_{i=1}^{n} L(y_i, \hat{y}_i) + \sum_{t=1}^{T} \Omega(f_t)$$
$$\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

- **γ (gamma)** — penalizes number of leaves → acts as pruning
- **λ (lambda)** — L2 penalty on leaf weights
- **η (learning_rate)** — shrinkage factor


In [ ]:
# Tune max_depth and n_estimators using a grid search approach
from sklearn.model_selection import GridSearchCV

cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Manual heatmap: max_depth vs n_estimators
depths = [2, 3, 4, 5, 6]
n_ests = [50, 100, 200, 300]
scores = np.zeros((len(depths), len(n_ests)))

for i, d in enumerate(depths):
    for j, n in enumerate(n_ests):
        m = GradientBoostingClassifier(max_depth=d, n_estimators=n,
                                        learning_rate=0.1, random_state=42)
        score = cross_val_score(m, X_tr, y_tr, cv=3, scoring='accuracy').mean()
        scores[i, j] = score

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(scores, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=n_ests, yticklabels=depths, ax=ax)
ax.set_xlabel('n_estimators', fontsize=12)
ax.set_ylabel('max_depth', fontsize=12)
ax.set_title('CV Accuracy Heatmap: max_depth vs n_estimators\n(Breast Cancer Dataset)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_idx = np.unravel_index(scores.argmax(), scores.shape)
print(f'Best: max_depth={depths[best_idx[0]]}, n_estimators={n_ests[best_idx[1]]}, CV accuracy={scores.max():.4f}')


In [ ]:
# Feature importance from GBM
best_model = GradientBoostingClassifier(
    max_depth=3, n_estimators=200, learning_rate=0.1,
    subsample=0.8, random_state=42
)
best_model.fit(X_tr, y_tr)

feat_importance = best_model.feature_importances_
feat_names = cancer.feature_names
sorted_idx = np.argsort(feat_importance)[-15:]  # top 15

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = plt.cm.YlOrRd(np.linspace(0.3, 1.0, len(sorted_idx)))
ax.barh(range(len(sorted_idx)), feat_importance[sorted_idx],
        color=colors_imp, edgecolor='k', alpha=0.85)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feat_names[i] for i in sorted_idx])
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Top 15 Feature Importances — Gradient Boosting\n(Breast Cancer)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 5. Final Comparison: All Three Algorithms

Let's train AdaBoost, GBM, and Random Forest (Bagging) on the same dataset and compare.


In [ ]:
# Side-by-side comparison on Breast Cancer
from sklearn.metrics import roc_curve, auc

models = {
    'Single Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                     max_depth=3, random_state=42),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, preds)
    fpr, tpr, _ = roc_curve(y_te, proba)
    roc_auc = auc(fpr, tpr)
    results[name] = {'acc': acc, 'auc': roc_auc, 'fpr': fpr, 'tpr': tpr}

# Bar chart — accuracy
names = list(results.keys())
accs  = [results[n]['acc']  for n in names]
aucs  = [results[n]['auc']  for n in names]
bar_colors = ['#94A3B8', '#16A34A', '#0D8E8E', '#F0A500']

bars = axes[0].bar(range(len(names)), accs, color=bar_colors, edgecolor='k', alpha=0.85)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=15, ha='right', fontsize=10)
axes[0].set_ylabel('Test Accuracy', fontsize=12)
axes[0].set_title('Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylim(0.88, 1.0)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{acc:.3f}', ha='center', fontweight='bold', fontsize=11)

# ROC curves
for (name, res), color in zip(results.items(), bar_colors):
    axes[1].plot(res['fpr'], res['tpr'], color=color, linewidth=2.5,
                 label=f"{name} (AUC={res['auc']:.3f})")
axes[1].plot([0,1], [0,1], 'k--', alpha=0.4, label='Random (AUC=0.5)')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curves', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)

plt.suptitle('Model Comparison on Breast Cancer Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Summary table
print('='*65)
print(f'{"Model":<30} {"Accuracy":>10} {"ROC-AUC":>10}')
print('='*65)
for name in names:
    print(f'{name:<30} {results[name]["acc"]:>10.4f} {results[name]["auc"]:>10.4f}')
print('='*65)
print()
print('Key Takeaways:')
print('  - Single tree = lowest accuracy (high bias)')
print('  - Random Forest = good (reduces variance via Bagging)')
print('  - AdaBoost = focuses on hard samples')
print('  - Gradient Boosting = best overall (reduces bias progressively)')


---
## Summary — Formula Cheat Sheet

```
ADABOOST:
  wᵢ = 1/N                                 initialize weights
  ε  = Σ wᵢ × 1[wrong]                     weighted error
  α  = 0.5 × ln((1−ε) / ε)                 learner weight
  wᵢ ← wᵢ × exp(−α × yᵢ × hₜ(xᵢ))          update weights
  ŷ  = sign(Σ αₜ × hₜ(x))                  final vote

GRADIENT BOOSTING:
  F₀(x) = mean(y)                           initial prediction
  rₜ    = y − Fₜ₋₁(x)                       residuals (for MSE loss)
  hₜ    = fit tree to rₜ                    tree on residuals
  Fₜ(x) = Fₜ₋₁(x) + η × hₜ(x)              additive update

XGBOOST OBJECTIVE:
  Obj = Σ L(yᵢ, ŷᵢ) + γT + λ/2 × Σwⱼ²
  γ = leaf penalty (pruning), λ = L2 on leaf weights
```
